<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module340/Lab8.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Lab 8 — VQE for a Small Materials-Model Hamiltonian
**Quantum Optimization and Simulation — VQE Laboratory Series**

Use a transverse-field Ising model to connect VQE with computational materials research.

**Suggested use:** 10–15 minute instructor demonstration followed by approximately one hour of independent work.

**Notebook style:** Most code is supplied. Complete the small items marked **YOUR TURN** and answer the reflection questions.

> Qiskit displays measured bitstrings as `q_(n-1)...q_0`. When orbital labels are written in the order `q0, q1, ...`, this notebook explicitly notes the convention.

## Learning objectives
- Apply VQE to a materials-model Hamiltonian rather than a molecule.
- Interpret competing interaction terms.
- Compare VQE with exact diagonalization.
- Study how the ground state changes as a model parameter changes.

We use a short transverse-field Ising chain:

\[
H=-J(Z_0Z_1+Z_1Z_2)-h(X_0+X_1+X_2).
\]

This is a simplified materials/condensed-matter model. It is not a complete model of an industrial metal, polymer, chemical, or coating, but it demonstrates the same ground-state-estimation workflow used in computational materials research.

In [ ]:
# Run once in a fresh Google Colab session.
%pip -q install "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-algorithms~=0.4" "qiskit-nature~=0.8"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_aer import AerSimulator

SEED = 123
SHOTS = 4096

def run_counts(qc, shots=SHOTS, noise_model=None):
    backend = AerSimulator(noise_model=noise_model)
    tqc = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=shots, seed_simulator=SEED).result()
    return result.get_counts()

def q0_first(qiskit_bits):
    return qiskit_bits.replace(" ", "")[::-1]

In [ ]:
from scipy.optimize import minimize
from qiskit.circuit import ParameterVector

## Part A — Build the Hamiltonian

In [ ]:
def ising_hamiltonian(J, h):
    return SparsePauliOp.from_list([
        ("IZZ", -J),
        ("ZZI", -J),
        ("IIX", -h),
        ("IXI", -h),
        ("XII", -h),
    ])

J = 1.0
h = 0.8
H = ising_hamiltonian(J, h)

print(H)
exact = np.linalg.eigvalsh(H.to_matrix())[0]
print("Exact ground-state energy:", exact)

## Part B — Hardware-efficient ansatz

In [ ]:
def materials_ansatz(params):
    qc = QuantumCircuit(3)

    # Layer 1: local rotations
    for q in range(3):
        qc.ry(params[q], q)

    # Entangling chain
    qc.cx(0,1)
    qc.cx(1,2)

    # Layer 2: more local rotations
    for q in range(3):
        qc.ry(params[3+q], q)

    return qc

## Part C — Exact expectation-value objective

In [ ]:
history = []

def exact_ansatz_energy(params, H):
    qc = materials_ansatz(params)
    psi = Statevector.from_instruction(qc)
    value = np.real(psi.expectation_value(H))
    history.append(value)
    return value

initial = np.zeros(6)

result = minimize(
    lambda p: exact_ansatz_energy(p, H),
    x0=initial,
    method="COBYLA",
    options={"maxiter":120, "rhobeg":0.5}
)

print("VQE energy:", result.fun)
print("Exact energy:", exact)
print("Error:", result.fun-exact)

plt.plot(history)
plt.xlabel("Objective evaluation")
plt.ylabel("Energy")
plt.title("Materials-model VQE convergence")
plt.show()

## Part D — Sweep the field strength

In [ ]:
field_values = np.linspace(0.0, 2.0, 9)
exact_energies = []
vqe_energies = []

for h in field_values:
    Hh = ising_hamiltonian(J=1.0, h=h)
    exact_energies.append(np.linalg.eigvalsh(Hh.to_matrix())[0])

    result_h = minimize(
        lambda p: exact_ansatz_energy(p, Hh),
        x0=np.zeros(6),
        method="COBYLA",
        options={"maxiter":80}
    )
    vqe_energies.append(result_h.fun)

plt.plot(field_values, exact_energies, "o-", label="Exact")
plt.plot(field_values, vqe_energies, "s--", label="VQE")
plt.xlabel("Transverse field h/J")
plt.ylabel("Ground-state energy")
plt.legend()
plt.show()

### YOUR TURN
For `h=0`, inspect the exact ground-state eigenvectors. Which computational-basis configurations dominate? Then compare with a large field, such as `h=2`.

## Reflection
1. What physical tendency is represented by the \(-JZZ\) terms?
2. What competing tendency is represented by the \(-hX\) terms?
3. Why is this lab relevant to materials modeling even though it is only a three-qubit toy model?

<details>
<summary><b>Instructor solution / suggested answer</b></summary>


    1. For positive \(J\), the \(-JZZ\) terms favor neighboring spins aligned in the Z direction.
    2. The transverse field favors alignment along the X direction, which competes with definite Z alignment and creates superposition.
    3. Materials physics frequently studies simplified effective Hamiltonians to isolate collective mechanisms. The same VQE workflow—prepare a trial state, measure energy terms, and optimize parameters—extends to larger and more realistic models.

</details>